**7. Control flow and logical operators with JIT**

When executing eagerly (outside of jit), JAX code works with Python control flow and logical operators just like Numpy code. Using control flow and logical operators with jit is more complicated.

In a nutshell, Python control flow and logical operators are evaluated at JIT compile time, such that the compiled function represents a single path through the control flow graph (logical operators affect the path via short-circuiting).

Paths dependent on the dtype or shape are okay, but will require recompilation when a value of a new dtype or shape is provided.

In [1]:
import jax
import jax.numpy as jnp

For example, this works:

In [3]:
from jax import jit
@jit
def f(x):
  for i in range(3):
    x = 2 * x
  return x

print(f(3))

24


So does this:

In [4]:
@jit
def g(x):
  y = 0.
  for i in range(x.shape[0]):
    y = y + x[i]
  return y

print(g(jnp.array([1., 2., 3.])))

6.0


But this doesn’t, at least by default:

In [5]:
@jit
def f(x):
  if x < 3:
    return 3. * x ** 2
  else:
    return -4 * x

# This will fail!
f(2)

TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function f at /tmp/ipykernel_8711/3402096563.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError

Neither does this:

In [6]:
@jit
def g(x):
  return (x > 0) and (x < 3)

# This will fail!
g(2)

TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function g at /tmp/ipykernel_8711/543860509.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError

When we jit-compile a function, we usually want to compile a version of the function that works for many different argument values, so that we can cache and reuse the compiled code. That way we don’t have to re-compile on each function evaluation.

The good news is that you can control this tradeoff yourself. By having jit trace on more refined abstract values, you can relax the traceability constraints. For example, using the `static_argnames` (or `static_argnums`) argument to `jit`, we can specify to trace on concrete values of some arguments. Here’s that example function again:

In [7]:
def f(x):
  if x < 3:
    return 3. * x ** 2
  else:
    return -4 * x

f = jit(f, static_argnames='x')

print(f(2.))

12.0


An example with a loop;

In [8]:
def f(x, n):
  y = 0.
  for i in range(n):
    y = y + x[i]
  return y

f = jit(f, static_argnames='n')

f(jnp.array([2., 3., 4.]), 2)

Array(5., dtype=float32)

**Structured control flow primitives**

There are more options for control flow in JAX. Say you want to avoid re-compilations but still want to use control flow that’s traceable, and that avoids un-rolling large loops. Then you can use these 4 structured control flow primitives:

* lax.cond differentiable
* lax.while_loop fwd-mode-differentiable
* lax.fori_loop fwd-mode-differentiable in general; fwd and rev-mode differentiable if endpoints are static.
* lax.scan differentiable

**Conditions**

In [9]:
from jax import lax

# Python version
def cond(pred, true_fun, false_fun, operand):
  if pred:
    return true_fun(operand)
  else:
    return false_fun(operand)

# JAX Version
operand = jnp.array([0.])
lax.cond(True, lambda x: x+1, lambda x: x-1, operand)
# --> array([1.], dtype=float32)
lax.cond(False, lambda x: x+1, lambda x: x-1, operand)
# --> array([-1.], dtype=float32)

Array([-1.], dtype=float32)

LAX also provides other functions for conditions:

In [10]:
# 1) Choose between two precomputed values of the same shape and dtype
# lax.select(pred, a, b)
a = jnp.arange(10) * 10
b = jnp.arange(10) * 5
lax.select(a[-1] > b[-1], a, b)

# 2) Like lax.cond, but pick between any number of branch functions
# lax.switch(index, branches, operand)
branches = [lambda x: x + 1, lambda x: x - 1, lambda x: x * 2]
lax.switch(0, branches, 10)

Array(11, dtype=int32, weak_type=True)

**`while_loop`**

In [11]:
# Python version
def while_loop(cond_fun, body_fun, init_val):
  val = init_val
  while cond_fun(val):
    val = body_fun(val)
  return val

# JAX Version
init_val = 0
cond_fun = lambda x: x < 10
body_fun = lambda x: x+1
lax.while_loop(cond_fun, body_fun, init_val)
# --> array(10, dtype=int32)

Array(10, dtype=int32, weak_type=True)

**`fori_loop`**

In [12]:
# Python version
def fori_loop(start, stop, body_fun, init_val):
  val = init_val
  for i in range(start, stop):
    val = body_fun(i, val)
  return val

# JAX Version
init_val = 0
start = 0
stop = 10
body_fun = lambda i,x: x+i
lax.fori_loop(start, stop, body_fun, init_val)
# --> array(45, dtype=int32)

Array(45, dtype=int32, weak_type=True)

**Logical operators**

`jax.numpy` provides `logical_and`, `logical_o`r, and `logical_no`t, which operate element-wise on arrays and can be evaluated under jit without recompiling. Like their Numpy counterparts, the binary operators do not short circuit. Bitwise operators (&, |, ~) can also be used with jit.

For example, consider a function that checks if its input is a positive even integer. The pure Python and JAX versions give the same answer when the input is scalar.

In [13]:
def python_check_positive_even(x):
  is_even = x % 2 == 0
  # `and` short-circults, so when `is_even` is `False`, `x > 0` is not evaluated.
  return is_even and (x > 0)

@jit
def jax_check_positive_even(x):
  is_even = x % 2 == 0
  # `logical_and` does not short circuit, so `x > 0` is always evaluated.
  return jnp.logical_and(is_even, x > 0)

print(python_check_positive_even(24))
print(jax_check_positive_even(24))

True
True


JAX version with `logical_and` returns elementwise valueswhen applied to an array.

In [14]:
x = jnp.array([-1, 2, 5])
print(jax_check_positive_even(x))

[False  True False]


Python logical operators error when applied to JAX arrays of more than one element, even without jit. This replicates NumPy’s behavior.

In [15]:
print(python_check_positive_even(x))

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

If you just want to apply grad to your python functions, without jit, you can use regular Python control-flow constructs with no problems, as if you were using Autograd (or Pytorch or TF Eager)

In [17]:
from jax import grad
def f(x):
  if x < 3:
    return 3. * x ** 2
  else:
    return -4 * x

print(grad(f)(2.))  # ok!
print(grad(f)(4.))  # ok!

12.0
-4.0
